# Advanced: Multi-Strategy Date Field Analysis with PAMOLA.CORE

**Goal**: Master all 3 date analysis strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Group-based change detection (track changes within ID groups)
- **Strategy 2**: UID consistency validation (detect person-level inconsistencies)
- **Strategy 3**: Age derivation and semantic validation (birth date analysis)
- Calculate advanced data quality metrics
- Analyze temporal patterns and anomalies
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of date validation concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/analyzers/
        └── 02_date_analyzer_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime, timedelta
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.date import DateOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 6 columns)
- Sample records (first 5 rows)
- Column statistics (types, ranges, unique counts)

**Dataset features:**
- Multiple date fields: birth_date, hire_date, snapshot_date
- Group identifiers: resume_id (for change detection)
- Person identifiers: UID (for consistency checks)
- Intentional anomalies for testing

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'date_analysis_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate base data
    base_date = datetime(2020, 1, 1)
    
    # Generate birth dates (1970-2000)
    birth_dates = [
        (base_date - timedelta(days=np.random.randint(7300, 18250))).strftime('%Y-%m-%d')
        for _ in range(n_records)
    ]
    
    # Generate hire dates (2010-2023)
    hire_dates = [
        (base_date + timedelta(days=np.random.randint(3650, 5475))).strftime('%Y-%m-%d')
        for _ in range(n_records)
    ]
    
    # Generate snapshot dates (2020-2024)
    snapshot_dates = [
        (base_date + timedelta(days=np.random.randint(0, 1460))).strftime('%Y-%m-%d')
        for _ in range(n_records)
    ]
    
    # Create resume_id with some duplicates (for group analysis)
    resume_ids = [f"R{i//3:04d}" for i in range(n_records)]
    
    # Create UID with some duplicates (for consistency checks)
    uids = [f"U{i//2:04d}" for i in range(n_records)]
    
    df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'resume_id': resume_ids,
        'UID': uids,
        'birth_date': birth_dates,
        'hire_date': hire_dates,
        'snapshot_date': snapshot_dates
    })
    
    # Introduce intentional anomalies for testing
    # 1. Some invalid birth dates (too old)
    df.loc[0:4, 'birth_date'] = '1920-01-01'
    # 2. Some future dates
    df.loc[5:9, 'hire_date'] = '2030-12-31'
    # 3. Some invalid formats
    df.loc[10:14, 'snapshot_date'] = 'invalid-date'
    # 4. Some inconsistent dates within same UID
    df.loc[df['UID'] == 'U0005', 'birth_date'] = ['1985-03-15', '1990-07-20']
    # 5. Some date changes within same resume_id
    df.loc[df['resume_id'] == 'R0010', 'hire_date'] = ['2015-01-10', '2015-06-15', '2015-12-20']
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

print(f"\n💡 Perfect dataset for testing all 3 strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field names for each strategy
   - `"strategy1_field"`: For group-based analysis (default: "hire_date")
   - `"strategy2_field"`: For UID consistency (default: "birth_date")
   - `"strategy3_field"`: For age derivation (default: "birth_date")
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each field)
- Unique value counts per field
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Note:** All fields must exist in dataset before executing strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_field": "hire_date",      # Group-based change detection
    "strategy2_field": "birth_date",     # UID consistency validation
    "strategy3_field": "birth_date"      # Age derivation & validation
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {strategy} not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    print(f"  ✓ {strategy:20s}: '{field_name}' ({df[field_name].nunique()} unique values)")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy",
    description="Multi-strategy date field analysis",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Group-Based Change Detection

**How to use:**
- Detects date value changes within the same group (resume_id)
- Useful for tracking updates, corrections, or data quality issues
- Run to execute group-based analysis

**Key parameters:**
- `field_name`: Date field to analyze (from FIELD_CONFIG)
- `id_column="resume_id"`: Group identifier for change detection
- `min_year=2010`, `max_year=2024`: Valid range for hire dates
- `is_birth_date=False`: Not analyzing birth dates
- `mode='ENRICH'`: Keeps original data

**What you'll see:**
- Configuration summary
- Progress: validation → parsing → group analysis → anomalies → metrics → save
- Completion time (2-5 seconds)
- Groups with changes detected

**Note:** Creates detailed change reports in output files

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: GROUP-BASED CHANGE DETECTION")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Group-based",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = DateOperation(
    field_name=FIELD_CONFIG["strategy1_field"],
    min_year=2010,
    max_year=2024,
    id_column="resume_id",
    uid_column=None,
    is_birth_date=False,
    profile_type='date',
    chunk_size=10000,
    use_dask=False,
    use_vectorization=False,
    parallel_processes=1,
    npartitions=2,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"  Field:            {operation_s1.field_name}")
print(f"  ID column:        {operation_s1.id_column}")
print(f"  Year range:       {operation_s1.min_year}-{operation_s1.max_year}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_group_detection',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load and display key results
output_files_s1 = sorted(
    list((task_dir / 'strategy1_group_detection' / 'output').glob('*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    with open(output_files_s1[0], 'r') as f:
        stats_s1 = json.load(f)
    
    if 'date_changes_within_group' in stats_s1:
        changes = stats_s1['date_changes_within_group']
        print(f"\n📊 Groups with date changes: {changes.get('groups_with_changes', 0)}")
        if changes.get('examples'):
            print(f"\n🔍 Example groups with changes:")
            for ex in changes['examples'][:3]:
                print(f"   Group {ex['group_id']}: {len(ex['date_values'])} different dates")

## STRATEGY 2: UID Consistency Validation

**How to use:**
- Validates date consistency across records for same person (UID)
- Detects data quality issues where person has multiple birth dates
- Run to execute UID-based consistency checks

**Key parameters:**
- `field_name`: Date field to validate (birth_date from FIELD_CONFIG)
- `uid_column="UID"`: Person identifier for consistency checks
- `min_year=1940`, `max_year=2005`: Valid birth year range
- `is_birth_date=True`: Enable birth date specific validation

**What you'll see:**
- Configuration summary
- Progress: validation → parsing → UID analysis → metrics → save
- Completion time (2-5 seconds)
- UIDs with inconsistent dates

**Note:** Critical for identifying data integrity issues

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: UID CONSISTENCY VALIDATION")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: UID consistency",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = DateOperation(
    field_name=FIELD_CONFIG["strategy2_field"],
    min_year=1940,
    max_year=2005,
    id_column=None,
    uid_column="UID",
    is_birth_date=True,
    profile_type='date',
    chunk_size=10000,
    use_dask=False,
    use_vectorization=False,
    parallel_processes=1,
    npartitions=2,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"  Field:            {operation_s2.field_name}")
print(f"  UID column:       {operation_s2.uid_column}")
print(f"  Year range:       {operation_s2.min_year}-{operation_s2.max_year}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_uid_consistency',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load and display key results
output_files_s2 = sorted(
    list((task_dir / 'strategy2_uid_consistency' / 'output').glob('*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    with open(output_files_s2[0], 'r') as f:
        stats_s2 = json.load(f)
    
    if 'date_inconsistencies_by_uid' in stats_s2:
        inconsistencies = stats_s2['date_inconsistencies_by_uid']
        print(f"\n📊 UIDs with inconsistencies: {inconsistencies.get('uids_with_inconsistencies', 0)}")
        if inconsistencies.get('examples'):
            print(f"\n🔍 Example UIDs with inconsistent dates:")
            for ex in inconsistencies['examples'][:3]:
                print(f"   UID {ex['uid']}: {len(ex['date_values'])} different dates")

## STRATEGY 3: Age Derivation & Semantic Validation

**How to use:**
- Derives ages from birth dates and validates semantic correctness
- Detects "too young" anomalies (field misused as snapshot date)
- Run to execute age-based analysis

**Key parameters:**
- `field_name`: Birth date field (from FIELD_CONFIG)
- `is_birth_date=True`: Enable age calculation and validation
- `min_year=1940`, `max_year=2005`: Valid birth year range
- `id_column=None`, `uid_column=None`: Pure date analysis

**What you'll see:**
- Configuration summary
- Progress: validation → age calculation → semantic validation → save
- Completion time (2-5 seconds)
- Age distribution statistics
- Semantic validation results

**Note:** High "too_young" anomaly ratio indicates field misuse

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: AGE DERIVATION & SEMANTIC VALIDATION")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Age validation",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = DateOperation(
    field_name=FIELD_CONFIG["strategy3_field"],
    min_year=1940,
    max_year=2005,
    id_column=None,
    uid_column=None,
    is_birth_date=True,
    profile_type='date',
    chunk_size=10000,
    use_dask=False,
    use_vectorization=False,
    parallel_processes=1,
    npartitions=2,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"  Field:            {operation_s3.field_name}")
print(f"  Is birth date:    {operation_s3.is_birth_date}")
print(f"  Year range:       {operation_s3.min_year}-{operation_s3.max_year}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_age_validation',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load and display key results
output_files_s3 = sorted(
    list((task_dir / 'strategy3_age_validation' / 'output').glob('*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    with open(output_files_s3[0], 'r') as f:
        stats_s3 = json.load(f)
    
    if 'age_statistics' in stats_s3:
        age_stats = stats_s3['age_statistics']
        print(f"\n📊 Age Statistics:")
        print(f"   Min age:    {age_stats.get('min_age')}")
        print(f"   Max age:    {age_stats.get('max_age')}")
        print(f"   Mean age:   {age_stats.get('mean_age', 0):.1f}")
        print(f"   Median age: {age_stats.get('median_age')}")
    
    if 'anomalies' in stats_s3:
        anomalies = stats_s3['anomalies']
        too_young = anomalies.get('too_young', 0)
        total = stats_s3.get('total_records', 1)
        ratio = too_young / total if total > 0 else 0
        
        print(f"\n⚠️  Too young anomalies: {too_young} ({ratio:.1%})")
        if ratio > 0.2:
            print(f"   ⚠️  High anomaly ratio - possible field misuse!")

## Step 4: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and data quality metrics

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Data quality comparison (valid rates, anomalies)
- Strategy-specific metrics summary

**Note:** Fastest execution doesn't always mean best - consider data quality outcomes

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Group detection): {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (UID consistency): {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Age validation):  {elapsed_s3:6.2f}s")
print(f"  Total:                        {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print(f"\n📈 Data Quality Metrics:")
strategies = [
    ('Strategy 1', output_files_s1),
    ('Strategy 2', output_files_s2),
    ('Strategy 3', output_files_s3)
]

for name, files in strategies:
    if files:
        with open(files[0], 'r') as f:
            stats = json.load(f)
        print(f"\n  {name}:")
        print(f"    Valid rate:     {stats.get('valid_rate', 0):.2f}%")
        print(f"    Anomalies:      {sum(stats.get('anomalies', {}).values())}")

## Step 5: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Focus on validation points per strategy

**What you'll see (per strategy):**
1. **Metrics JSON**: Performance, effectiveness, data quality
2. **Statistics Data**: Distributions, anomalies, analysis results
3. **Visualizations**: Charts displayed inline (latest batch only)

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_group_detection', 'Strategy 1: Group-Based Detection'),
    ('strategy2_uid_consistency', 'Strategy 2: UID Consistency'),
    ('strategy3_age_validation', 'Strategy 3: Age Validation')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics Summary (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"\n✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"\n⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # Display key metrics
                print("   " + "-" * 60)
                for key in ['total_rows', 'total_records', 'valid_count', 'invalid_count', 
                           'fill_rate', 'valid_rate', 'anomalies_count']:
                    if key in metrics:
                        value = metrics[key]
                        if isinstance(value, float):
                            print(f"   {key:20s}: {value:.2f}%")
                        else:
                            print(f"   {key:20s}: {value:,}")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Detailed Statistics (from output/)
    output_dir = strategy_dir / 'output'
    if output_dir.exists():
        stat_files = sorted(
            list(output_dir.glob('*_stats_*.json')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if stat_files:
            print(f"\n📊 DETAILED STATISTICS: {stat_files[0].name}")
            print("   " + "-" * 60)
            
            try:
                with open(stat_files[0], 'r') as f:
                    stats = json.load(f)
                
                # Display key statistics
                key_stats = [
                    'total_records', 'null_count', 'non_null_count',
                    'valid_count', 'invalid_count', 'fill_rate', 'valid_rate',
                    'min_date', 'max_date'
                ]
                
                for key in key_stats:
                    if key in stats:
                        value = stats[key]
                        if isinstance(value, float):
                            print(f"   {key:20s}: {value:.2f}")
                        else:
                            print(f"   {key:20s}: {value}")
                
                # Show anomaly summary
                if 'anomalies' in stats:
                    print(f"\n   🔍 Anomaly Summary:")
                    for anomaly_type, count in stats['anomalies'].items():
                        if count > 0:
                            print(f"      • {anomaly_type:20s}: {count:,} records")
                
                # Show age statistics if available
                if 'age_statistics' in stats:
                    print(f"\n   📈 Age Statistics:")
                    age_stats = stats['age_statistics']
                    for key, value in age_stats.items():
                        if isinstance(value, float):
                            print(f"      • {key:15s}: {value:.1f}")
                        else:
                            print(f"      • {key:15s}: {value}")
                
            except Exception as e:
                print(f"   ⚠️  Error reading statistics: {e}")
    
    # 3. Anomaly Data (from dictionaries/)
    dict_dir = strategy_dir / 'dictionaries'
    if dict_dir.exists():
        anomaly_files = sorted(
            list(dict_dir.glob('*_anomalies_*.csv')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if anomaly_files:
            print(f"\n🔍 ANOMALY DETAILS: {anomaly_files[0].name}")
            print("   " + "-" * 60)
            
            try:
                anomalies_df = pd.read_csv(anomaly_files[0])
                
                if len(anomalies_df) > 0:
                    # Show anomaly type distribution
                    if 'anomaly_type' in anomalies_df.columns:
                        type_counts = anomalies_df['anomaly_type'].value_counts()
                        print(f"   Total anomalies: {len(anomalies_df):,}")
                        print(f"\n   By Type:")
                        for anom_type, count in type_counts.items():
                            print(f"      • {anom_type:20s}: {count:,} ({count/len(anomalies_df)*100:.1f}%)")
                    
                    # Show sample records
                    print(f"\n   Sample Records (first 5):")
                    display_cols = [col for col in anomalies_df.columns if col in ['index', 'value', 'anomaly_type', 'year']]
                    sample = anomalies_df[display_cols].head(5)
                    
                    for idx, row in sample.iterrows():
                        print(f"      [{idx}] ", end="")
                        print(" | ".join([f"{col}: {row[col]}" for col in display_cols]))
                else:
                    print(f"   ✓ No anomalies detected")
                    
            except Exception as e:
                print(f"   ⚠️  Error reading anomaly data: {e}")
    
    # 4. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"\n   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Calculate Data Quality Metrics

**How to use:**
- Run AFTER all strategies complete
- Calculates comprehensive data quality scores

**What you'll see:**
- Overall data quality score (0-100)
- Component scores:
  - Completeness (fill rate)
  - Validity (format correctness)
  - Consistency (UID/group alignment)
  - Semantic correctness (age validity)

**Quality targets:**
- Score ≥ 90: Excellent quality
- Score ≥ 75: Good quality
- Score < 75: Needs improvement

**Note:** Requires all 3 strategies to have completed successfully

In [ ]:
print("\n" + "=" * 80)
print("📊 DATA QUALITY METRICS")
print("=" * 80 + "\n")

try:
    # Load statistics from all strategies
    with open(output_files_s1[0], 'r') as f:
        stats_s1 = json.load(f)
    with open(output_files_s2[0], 'r') as f:
        stats_s2 = json.load(f)
    with open(output_files_s3[0], 'r') as f:
        stats_s3 = json.load(f)
    
    # Calculate component scores
    completeness_score = (stats_s1.get('fill_rate', 0) + stats_s2.get('fill_rate', 0) + stats_s3.get('fill_rate', 0)) / 3
    validity_score = (stats_s1.get('valid_rate', 0) + stats_s2.get('valid_rate', 0) + stats_s3.get('valid_rate', 0)) / 3
    
    # Consistency score (based on inconsistencies)
    total_records = stats_s2.get('total_records', 1)
    inconsistencies = stats_s2.get('date_inconsistencies_by_uid', {}).get('uids_with_inconsistencies', 0)
    consistency_score = max(0, 100 - (inconsistencies / total_records * 100)) if total_records > 0 else 0
    
    # Semantic correctness (based on too_young anomalies)
    too_young = stats_s3.get('anomalies', {}).get('too_young', 0)
    semantic_score = max(0, 100 - (too_young / total_records * 100)) if total_records > 0 else 0
    
    # Overall quality score
    overall_score = (completeness_score + validity_score + consistency_score + semantic_score) / 4
    
    print(f"🎯 OVERALL DATA QUALITY: {overall_score:.1f}/100")
    print(f"\n📈 Component Scores:")
    print(f"   Completeness:         {completeness_score:.1f}% (fill rate)")
    print(f"   Validity:             {validity_score:.1f}% (format correctness)")
    print(f"   Consistency:          {consistency_score:.1f}% (UID alignment)")
    print(f"   Semantic Correctness: {semantic_score:.1f}% (age validity)")
    
    print(f"\n💡 Quality Assessment:")
    if overall_score >= 90:
        print("   ✅ Excellent - Data is production-ready")
    elif overall_score >= 75:
        print("   ✓ Good - Minor issues to address")
    else:
        print("   ⚠️  Needs Improvement - Review anomalies")
        
except Exception as e:
    print(f"⚠️  Error calculating metrics: {e}")
    print("   Ensure all 3 strategies completed successfully")

## Step 7: Export Analysis Results

**How to use:**
- Run AFTER all strategies complete
- Exports comprehensive analysis reports

**What you'll see (per strategy):**
- Export confirmation with file path
- Summary statistics
- Key findings preview

**Output structure:**
```
advanced_output/
├── strategy1_group_detection/
│   ├── output/*.json
│   ├── metrics/*.json
│   └── visualizations/*.png
├── strategy2_uid_consistency/
│   └── ...
└── strategy3_age_validation/
    └── ...
```

**Note:** All files include timestamps for version tracking

In [ ]:
print("=" * 80)
print("📦 EXPORTING ANALYSIS RESULTS")
print("=" * 80)

print(f"\n📂 Export base directory: {task_dir}\n")

# Summary for each strategy
strategies = [
    ('STRATEGY 1: GROUP-BASED DETECTION', 'strategy1_group_detection', output_files_s1),
    ('STRATEGY 2: UID CONSISTENCY', 'strategy2_uid_consistency', output_files_s2),
    ('STRATEGY 3: AGE VALIDATION', 'strategy3_age_validation', output_files_s3)
]

for title, dir_name, output_files in strategies:
    print("=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    strategy_dir = task_dir / dir_name
    
    if output_files:
        print(f"\n✅ Statistics: {output_files[0].name}")
        print(f"   Location: {strategy_dir / 'output'}")
        
        # Load and show summary
        with open(output_files[0], 'r') as f:
            stats = json.load(f)
        
        print(f"\n📈 Summary:")
        print(f"   Total records:  {stats.get('total_records', 0):,}")
        print(f"   Valid records:  {stats.get('valid_count', 0):,}")
        print(f"   Valid rate:     {stats.get('valid_rate', 0):.2f}%")
        print(f"   Total anomalies: {sum(stats.get('anomalies', {}).values())}")
    
    # Check for other artifacts
    metrics_files = list((strategy_dir / 'metrics').glob('*.json')) if (strategy_dir / 'metrics').exists() else []
    viz_files = list((strategy_dir / 'visualizations').glob('*.png')) if (strategy_dir / 'visualizations').exists() else []
    
    if metrics_files:
        print(f"\n✅ Metrics: {len(metrics_files)} file(s)")
    if viz_files:
        print(f"✅ Visualizations: {len(viz_files)} file(s)")
    
    print()

print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 All files saved to: {task_dir}")

if 'elapsed_s1' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total processing time: {total_time:.2f}s")

print("\n" + "=" * 80)
print("🎉 ANALYSIS COMPLETE!")
print("=" * 80)

## 🎯 Summary

**Accomplished:**
- ✅ 3 strategies implemented and compared
- ✅ Data quality metrics calculated
- ✅ Group-based change detection
- ✅ UID consistency validation
- ✅ Age derivation and semantic validation
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Group detection identifies temporal changes in date fields
- UID consistency reveals data integrity issues
- Semantic validation catches field misuse
- Combined analysis provides comprehensive quality assessment

**Strategy recommendations:**
- **Use Strategy 1** when: Tracking updates or changes over time
- **Use Strategy 2** when: Validating person-level data consistency
- **Use Strategy 3** when: Analyzing birth dates or age-dependent logic

**Next steps:**
- Test with your own datasets
- Adjust validation thresholds for your domain
- Deploy to production environment
- Integrate with data quality pipelines

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)